In [1]:
## init mongo db and fiftyone connection
import os

# Define the URI to point to your manual process
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost:44123"

import fiftyone as fo

# Verify connection
print(fo.core.odm.database.get_db_conn()) 


You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [2]:
import fiftyone.brain as fob
from sklearn.preprocessing import normalize
import plotly.express as px
import skdim
import random
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import ot
from sklearn.manifold import TSNE
import cv2
from fiftyone import ViewField as F
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import random
import glob 

# ## renders plotly properly in a html instance. 
# import plotly.io as pio
# pio.renderers.default = "notebook"


In [3]:
## Load dataset and views from mongodb 
dataset = fo.load_dataset("dugong")

## load the views
nc_view = dataset.load_saved_view("New_Caledonia")
wp_view = dataset.load_saved_view("West_Papua")

In [4]:
view_GAM = wp_view.match(
    F("subregion")=="GAM"
)
view_FRIWEN = wp_view.match(
    F("subregion")=="FRIWEN"
)
view_MANTASANDY = wp_view.match(
    F("subregion")=="MANTASANDY"
)
view_UM = wp_view.match(
    F("subregion")=="UM"
)

print("Number of images per geographical location")
print(f"WP- GAM: {len(view_GAM)}")
print(f"WP - FRIWEN: {len(view_FRIWEN)}")
print(f"WP - MANTASANDY: {len(view_MANTASANDY)}")
print(f"WP -UM: {len(view_UM)}")
print(f"NC: {len(nc_view)}")
print(f"WP (total): {len(wp_view)}")


Number of images per geographical location


WP- GAM: 512
WP - FRIWEN: 779
WP - MANTASANDY: 3
WP -UM: 745
NC: 716
WP (total): 2039


In [12]:
dict_wp = {
    'um':view_UM,
    'gam':view_GAM,
    'mantasandy':view_MANTASANDY,
    'friwen':view_FRIWEN
}

for name, dd in dict_wp.items():
    ## count the number of objects
    print(f"Running:{name}")

    number_of_objects = dd.count("ground_truth.detections")
    print(f"Number of objects:{number_of_objects}")

    # Filter detections in the 'audit' field
    medium_complexity_bg = dd.filter_labels(
        "audit", 
        F("background_complexity") == "medium"
    )

    high_complexity_bg = dd.filter_labels(
        "audit", 
        F("background_complexity") == "high"
    )

    low_complexity_bg = dd.filter_labels(
        "audit", 
        F("background_complexity") == "low"
    )

    # Check how many detections matched
    print(f"Number of images as:")
    high = dd.match(F("background_complexity") == "high")
    medium = dd.match(F("background_complexity") == "medium")
    low = dd.match(F("background_complexity") == "low")
    print(f"high: {high.count()}")
    print(f"medium: {medium.count()}")
    print(f"low: {low.count()}")
    print('- - '*2)
    print(f"OBJECTS")
    print('high complexity',high_complexity_bg.count("audit.detections"))
    print("medium complexity",medium_complexity_bg.count("audit.detections"))
    print("low complexity",low_complexity_bg.count("audit.detections"))
    print('-'*30)

Running:um
Number of objects:1308
Number of images as:
high: 4
medium: 741
low: 0
- - - - 
OBJECTS
high complexity 6
medium complexity 1302
low complexity 0
------------------------------
Running:gam
Number of objects:997
Number of images as:
high: 299
medium: 61
low: 152
- - - - 
OBJECTS
high complexity 571
medium complexity 122
low complexity 304
------------------------------
Running:mantasandy
Number of objects:3
Number of images as:
high: 2
medium: 1
low: 0
- - - - 
OBJECTS
high complexity 2
medium complexity 1
low complexity 0
------------------------------
Running:friwen
Number of objects:1307
Number of images as:
high: 4
medium: 775
low: 0
- - - - 
OBJECTS
high complexity 0
medium complexity 1302
low complexity 0
------------------------------


## Strategy for Splitting between train and test 

### Per-Site Stratified Random Splitting

Per-Site:  each geographical island is an independent unit, to prevent spatial bias.

Stratified: maintaining the ratio of "High/Medium/Low" complexity in every set.

Random Splitting: stochastic selection to ensure generalizability.

In [ ]:
from fiftyone import ViewField as F
import fiftyone.utils.random as four

# create a combined key: e.g., "WP_UM_medium" or "WP_GAM_high"
# ensures the split respects both the location and the difficulty
dataset.set_values(
    "stratify_key",
    [f"{r}_{m}_{c}" for r, m, c in zip(
        dataset.values("region"), 
        dataset.values("subregion"), 
        dataset.values("background_complexity")
    )]
)

print("Unique strata created:", dataset.count_values("stratify_key"))

Unique strata created: {'WP_MANTASANDY_medium': 1, 'WP_MANTASANDY_high': 2, 'WP_GAM_high': 299, 'WP_GAM_medium': 61, 'NC_NC_low': 108, 'WP_GAM_low': 152, 'NC_NC_medium': 270, 'NC_NC_high': 338, 'WP_FRIWEN_medium': 775, 'WP_UM_high': 4, 'WP_FRIWEN_high': 4, 'WP_UM_medium': 741}


In [18]:
dataset.count_values("subregion")

{'MANTASANDY': 3, 'FRIWEN': 779, 'GAM': 512, 'NC': 716, 'UM': 745}

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

## percentage of each train, val, test
train_size = 0.7
test_size = 0.2 
val_size = 0.1
random_state = 42

# target islands (skipping MANTASANDY)
islands_to_split = ['UM', 'GAM', 'FRIWEN']

# clear old tags to start fresh
dataset.untag_samples(["train", "test", "val"])

for island in islands_to_split:
    # Get a view of just this island
    island_view = dataset.match(F("subregion") == island)
    
    ids = island_view.values("id")

    # Our strata is the complexity (high/medium/low)
    strata = island_view.values("background_complexity")
    
    # Split off the TEST set (20%) ---
    # Stratify ensures the 'high complexity' ratio stays the same
    train_val_ids, test_ids = train_test_split(
        ids, 
        test_size= test_size, 
        stratify=strata,
        shuffle=True, 
        random_state=random_state
    )
    
    # Get strata for the remaining 80% to split again
    train_val_strata = [s for i, s in zip(ids, strata) if i in train_val_ids]
    
    # Split remaining 80% into Train (70% total) and Val (10% total) ---
    # 0.125 * 0.8 = 0.1 (which is 10% of the original total)
    train_ids, val_ids = train_test_split(
        train_val_ids, 
        test_size= (val_size/(1-test_size)), 
        stratify=train_val_strata, 
        random_state=random_state
    )
    
    # 3. Apply the tags in FiftyOne
    dataset.select(train_ids).tag_samples("train")
    dataset.select(val_ids).tag_samples("val")
    dataset.select(test_ids).tag_samples("test")
    
    print(f"Island {island}: Train={len(train_ids)}, Test={len(test_ids)}, Val={len(val_ids)}")

# 4. Add New Caledonia to 'train' only
nc_view = dataset.match(F("region") == "NC")
nc_view.tag_samples("train")

print('dataset split:')
print(f"train {wp_view.count_values('tags')['train'] + nc_view.count_values('tags')['train']}")
print('test',wp_view.count_values('tags')['test'])
print('val',wp_view.count_values('tags')['val'])

Island UM: Train=521, Test=149, Val=75
Island GAM: Train=357, Test=103, Val=52
Island FRIWEN: Train=545, Test=156, Val=78


### Repeated Random Sub-sampling Validation

In [ ]:
# clear old tags to start fresh
dataset.untag_samples(["train", "test", "val"])


# collect all tags in dataset
all_tags = dataset.distinct("tags")

# keep only those starting with "train_"
tags_to_remove = [t for t in all_tags if t.startswith("train_")]

# delete them
dataset.untag_samples(tags_to_remove)

In [14]:
from sklearn.model_selection import train_test_split
import pandas as pd


def tag_train_test_split_seeded(train_size:float,test_size:float,val_size:float,
                                runs:int,
                                dataset):
    """
    Creates the split for train, test and validation using a stratified pick given by the complexity. 
    Args:
        runs: Number of loop to pass and create the tag inside the dataset 
    
        Returns:
        Return the seed_number and the dataset get tagged.
    """
    # target islands (skipping MANTASANDY)
    islands_to_split = ['UM', 'GAM', 'FRIWEN']
    seeds_list = []

    ## tags adding train_seed_number for west papua 
    for run in range(0,runs+1):
        seed_number  = random.randint(1,50)
        print(f"Seed number selected:{seed_number}")
        seeds_list.append(seed_number)
        for island in islands_to_split:
            # Get a view of just this island
            island_view = dataset.match(F("subregion") == island)
            
            ids = island_view.values("id")

            # Our strata is the complexity (high/medium/low)
            strata = island_view.values("background_complexity")
            
            # Split off the TEST set (20%) ---
            # Stratify ensures the 'high complexity' ratio stays the same
            train_val_ids, test_ids = train_test_split(
                ids, 
                test_size= test_size, 
                stratify=strata,
                shuffle=True, 
                random_state=seed_number
            )
            
            # Get strata for the remaining 80% to split again
            train_val_strata = [s for i, s in zip(ids, strata) if i in train_val_ids]
            
            # Split remaining 80% into Train (70% total) and Val (10% total) ---
            # 0.125 * 0.8 = 0.1 (which is 10% of the original total)
            train_ids, val_ids = train_test_split(
                train_val_ids, 
                test_size= (val_size/(1-test_size)), 
                stratify=train_val_strata, 
                random_state=seed_number
            )
            
            # 3. Apply the tags in FiftyOne
            dataset.select(train_ids).tag_samples(f"train_{str(seed_number)}")
            dataset.select(val_ids).tag_samples(f"val_{str(seed_number)}")
            dataset.select(test_ids).tag_samples(f"test_{str(seed_number)}")
            
            print(f"Island {island}: Train={len(train_ids)}, Test={len(test_ids)}, Val={len(val_ids)}")

    return seeds_list


def get_files_by_stem(filepath_stem, folder):
    dict_out = {}
    foolder_meta = os.path.join(folder, 'metadata')
    list_meta = list(glob.glob(os.path.join(foolder_meta, f'{filepath_stem}__*.json')))
    foolder_meta = os.path.join(folder, 'images')
    list_images = list(glob.glob(os.path.join(foolder_meta, f'{filepath_stem}__*.jpg')))
    foolder_meta = os.path.join(folder, 'labels')
    list_labels = list(glob.glob(os.path.join(foolder_meta, f'{filepath_stem}__*.txt')))
    dict_out['metadata'] = list_meta
    dict_out['label'] = list_labels
    dict_out['images'] = list_images
    return dict_out


## use the function to run 
folder =  '/share/home/e2406743/dataset/exported_img/seed_42'
## percentage of each train, val, test
train_size = 0.7
test_size = 0.2 
val_size = 0.1
runs = 1

seed_number_list = tag_train_test_split_seeded(train_size, test_size, val_size,
                                          runs=runs,
                                          dataset= dataset
                                          )

## tag TRAIN for all new caledonia samples.
nc_view = dataset.match(F("region") == "NC")
nc_view.tag_samples("train")


Seed number selected:23
Island UM: Train=521, Test=149, Val=75
Island GAM: Train=357, Test=103, Val=52
Island FRIWEN: Train=545, Test=156, Val=78
Seed number selected:11
Island UM: Train=521, Test=149, Val=75
Island GAM: Train=357, Test=103, Val=52
Island FRIWEN: Train=545, Test=156, Val=78


In [22]:
def return_list_filepath_train_test_val(seed_number, dataset, nc_view):
    train_seed_filepath = dataset.match_tags(f"train_{seed_number}").values("filepath")
    test_seed_filepath = dataset.match_tags(f"test_{seed_number}").values("filepath")
    val_seed_filepath = dataset.match_tags(f"val_{seed_number}").values("filepath")
    train_nc_filepath = nc_view.match_tags("train").values("filepath")
    return train_seed_filepath, test_seed_filepath, val_seed_filepath, train_nc_filepath


def build_filepath_df(train_seed_filepath, test_seed_filepath, val_seed_filepath, train_nc_filepath):

    df = pd.DataFrame({
        "train_seed": pd.Series(train_seed_filepath),
        "test_seed": pd.Series(test_seed_filepath),
        "val_seed": pd.Series(val_seed_filepath),
        "train_nc": pd.Series(train_nc_filepath),
    })

    return df



## IMPLEMENT LOOP HERE
## RUN ALL GIVEN SEEDS AND CREATES A CSV WITH THE PATHS REGARDING THE FULL IMAGE
for ss in seed_number_list:
    print(f"Running seed:{ss}")
    train_seed_filepath, test_seed_filepath, val_seed_filepath, train_nc_filepath = return_list_filepath_train_test_val(
        ss, dataset=dataset, nc_view=nc_view
    )

    df_seed = build_filepath_df(
        train_seed_filepath,
        test_seed_filepath,
        val_seed_filepath,
        train_nc_filepath
    )

    ## save it keeping 
    output_filename = f"df_train_test_split_filepath_{str(ss)}"
    print(f"saving file:{output_filename}")
    output_folder = "/share/home/e2406743/dataset/df_filepaths"
    os.makedirs(output_folder, exist_ok=True)
    print(f"saving at:{os.path.join(output_folder,output_filename)}")
    df_seed.to_csv(os.path.join(output_folder, output_filename))
    print('done!')


Running seed:23
saving file:df_train_test_split_filepath_23
saving at:/share/home/e2406743/dataset/df_filepaths/df_train_test_split_filepath_23
done!
Running seed:11
saving file:df_train_test_split_filepath_11
saving at:/share/home/e2406743/dataset/df_filepaths/df_train_test_split_filepath_11
done!


In [ ]:
## LOAD THE DF WITH PATHS AND PREPARE TO BE SAMPLED

205

In [9]:
vv = wp_view.values('filepath')
ss = nc_view.values('filepath')
dd = vv +ss 

folder =  '/share/home/e2406743/dataset/exported_img/seed_42'
dict_map_filepath = {}
for path in dd:
    stem = Path(path).stem
    dict_map_filepath[stem] = get_files_by_stem(stem, folder)

## flat dict
filepath_all_images   = [f for d in dict_map_filepath.values() for f in d.get('images', [])]
filepath_all_labels   = [f for d in dict_map_filepath.values() for f in d.get('label', [])]
filepath_all_metadata = [f for d in dict_map_filepath.values() for f in d.get('metadata', [])]

## -------------
print(f'images:{len(filepath_all_images)}')
print(f"labels:{len(filepath_all_labels)}")
print(f"metadata:{len(filepath_all_metadata)}")

images:10008
labels:10008
metadata:10008


In [ ]:
len()

In [ ]:
# list with NC train
# list with WP train 10%
# list with WP train 20% ... 70% 
# list with selected optimization 

# Structure model

In [5]:
import os 
from dotenv import load_dotenv
load_dotenv()

import fiftyone as fo
import fiftyone.utils.torch as fout
from huggingface_hub import login

if "HUGGING_FACE_API" in os.environ:
    login(token=os.environ["HUGGING_FACE_API"])

In [28]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import os
from pathlib import Path

class Dataset_patched_Dugong(Dataset):
    def __init__(self,list_image_filepath,
                 list_label_filepath,
                   processor):
        """
        Args:
            processor: HuggingFace RTDetrImageProcessor
        """
        self.list_image_filepath =sorted(list_image_filepath)
        self.list_label_filepath =sorted(list_label_filepath)
        assert len(self.list_image_filepath) == len(self.list_label_filepath)
        self.processor = processor

    def __len__(self):
        return len(self.list_image_filepath)
    def __getitem__(self, idx):
        
        # Determine paths based on your export structure
        patch_path = self.list_image_filepath[idx]
        label_path = self.list_label_filepath[idx]
        
        # 2. Load Image
        image = Image.open(patch_path).convert("RGB")
        
        # 3. Parse YOLO Labels (.txt)
        annotations = []
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                lines = f.readlines()
            
            for line in lines:
                parts = line.strip().split()
                if not parts: continue
                
                # YOLO format: class_id, x_center, y_center, width, height
                cls_id, xc, yc, w, h = map(float, parts)
                
                annotations.append({
                    "class_id": int(cls_id),
                    "bbox": [xc, yc, w, h], # YOLO format (normalized center)
                    "area": w * h,
                    "iscrowd": 0
                })

        # 4. Process for Transformer
        # We tell the processor the boxes are in 'yolo' format so it 
        # doesn't try to convert them from COCO top-left.
        encoding = self.processor(
            images=image, 
            annotations={"image_id": idx, "annotations": annotations}, 
            return_tensors="pt"
        )
        
        # Squeeze out batch dimension added by the processor
        pixel_values = encoding["pixel_values"].squeeze(0)
        labels = encoding["labels"][0]
        
        return pixel_values, labels

def collate_fn(batch):
    return {
        "pixel_values": torch.stack([item[0] for item in batch]),
        "labels": [item[1] for item in batch]
    }

In [ ]:

from transformers import RTDetrForObjectDetection, RTDetrImageProcessor
from torch.utils.data import DataLoader

# 1. Setup Model & Processor
checkpoint = "PekingU/rtdetr_r50vd"
processor = RTDetrImageProcessor.from_pretrained(checkpoint)
model = RTDetrForObjectDetection.from_pretrained(checkpoint)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 2. Create Dataloaders (Pass your lists of JSON paths here)
train_dataset = RTDetrDataset(filepath_all_images, filepath_all_labels, processor)

preprocessor_config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/172M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/764 [00:00<?, ?it/s]

NameError: name 'torch' is not defined

In [29]:
dataset = Dataset_patched_Dugong(filepath_all_images, filepath_all_labels, processor)

In [30]:
train_loader = DataLoader(dataset,
                           batch_size=8,
                             shuffle=True,
                               collate_fn=collate_fn)

In [ ]:
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor
from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch
import torch.nn as nn
import kornia.augmentation as K
import random

seed =0
## seed everything
# torch.manual_seed(0)
# random.seed(0)
# np.random.sedd
# torch.backends.cudnn.benchmark == False


class DugongAugmentor(nn.Module):
    def __init__(self):
        super().__init__()
        # We define a sequence of augmentations
        # data_keys=["input", "bbox"] tells Kornia to transform the boxes too!
        self.augmentations = K.AugmentationSequential(
            K.RandomHorizontalFlip(p=0.5),
            K.RandomVerticalFlip(p=0.5),
            K.RandomRotation(degrees=15.0, p=0.3),
            K.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
            data_keys=["input", "bbox"], 
        )

    @torch.no_grad()
    def forward(self, images, boxes):
        """
        images: (B, 3, 640, 640)
        boxes: List of Tensors or a padded Tensor
        """
        # Kornia expects boxes as [B, N, 4] or [B, N, 4, 2]
        # We transform them from [cx, cy, w, h] to [x1, y1, x2, y2] for Kornia
        # Then back again for RT-DETR
        return self.augmentations(images, boxes)
  

# 1. Setup Model & Processor
checkpoint = "PekingU/rtdetr_r50vd"

## create processor for transformers library 
processor = RTDetrImageProcessor.from_pretrained(checkpoint)
model = RTDetrForObjectDetection.from_pretrained(checkpoint)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Dataset
train_dataset = dataset = Dataset_patched_Dugong(filepath_all_images, filepath_all_labels, processor)

# Dataloader
train_loader = DataLoader(train_dataset,
                           batch_size=8,
                             shuffle=True,
                               collate_fn=collate_fn)

# Optimizer
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)


#  Augmentor
augmentor = DugongAugmentor().to(device)

model.train()
for batch in train_loader:
    optimizer.zero_grad()
    
    # Move to GPU
    pixel_values = batch["pixel_values"].to(device)
    
    # RT-DETR boxes are in a list of dicts. 
    # To use Kornia, we extract them into a single Tensor [B, Max_Detections, 4]
    # Note: For simplicity, assuming 1 detection per tile for this snippet.
    # In a full thesis loop, you would pad these to a fixed size.
    gt_boxes = torch.stack([l["boxes"] for l in batch["labels"]]).to(device)
    
    # 2. APPLY KORNIA (GPU-accelerated)
    # This transforms both pixels and boxes simultaneously
    pixel_values_aug, gt_boxes_aug = augmentor(pixel_values, gt_boxes)
    
    # 3. Update the labels dict with the new augmented boxes
    for i, label in enumerate(batch["labels"]):
        label["boxes"] = gt_boxes_aug[i]
    
    # 4. Forward Pass
    outputs = model(pixel_values=pixel_values_aug,
                        labels=batch["labels"])
    
    loss = outputs.loss
    loss.backward()
    optimizer.step()

In [ ]:
# After training, load your best weights
model = RTDETR("./Dugong_Thesis/RTDETR_Run1/weights/best.pt")

# Select the test view to see how it performed on the unseen West Papua data
test_view = dataset.match_tags("test_run1")

# Apply the model to generate a new field: 'predictions_rtdetr'
test_view.apply_model(model, label_field="predictions_rtdetr")

# Launch the App to visualize the 'Audit' vs 'Predictions'
session = fo.launch_app(dataset)